# Liu2024 Source MAT SignalJEPA PreLocal - Leave-One-Subject-Out (cross-subject)

Cross-subject evaluation for S-JEPA PreLocal on the Liu2024 acute-stroke MI dataset.

Instead of the within-subject 5-fold protocol (32 train / 8 test trials, which collapses),
this notebook does **leave-one-subject-out (LOSO)**: for each held-out test subject, the model
trains on the other 49 subjects (~1,960 trials) with a **subject-level** validation holdout so
early stopping is stable. This is the data regime in which a pretrained encoder can actually adapt.

Structure mirrors the within-subject notebook: Setup -> Configuration (CONFIG) -> Data -> Model ->
LOSO splits -> Training/eval runner -> Run loop -> Aggregate.

# 1. Setup

In [1]:
import os
import re
import sys
import json
import math
import random
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import torch

from scipy.io import loadmat

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from skorch.callbacks import EarlyStopping
from skorch.helper import predefined_split

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal

import mne
mne.set_log_level("WARNING")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Runtime Environment:")
print(f"  - Python:   {sys.version.split()[0]}")
print(f"  - Platform: {platform.platform()}")
print(f"  - Torch:    {torch.__version__}")


Runtime Environment:
  - Python:   3.11.15
  - Platform: macOS-26.5.1-arm64-arm-64bit
  - Torch:    2.10.0


# 2. Configuration

## 2.1 Liu2024 channel defaults

Source `.mat` files are `trials x 33 channels x samples`:
indices 0..29 are EEG-like, index 17 is the CPz source reference, 30..31 are EOG, 32 is the marker.
We keep the 29 EEG channels used by the Liu paper baseline and drop CPz (reference), EOG and marker.

In [3]:
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32

EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
                     if idx != SOURCE_REFERENCE_INDEX]


## 2.2 CONFIG

Same style as the within-subject notebook. The LOSO-specific keys are grouped under
"Evaluation protocol". Note the defaults changed deliberately:
- `mi_window_start_s = 2.0`  -> aligned to the MI cue (trial = instruction 0-2 s, MI 2-6 s, break 6-8 s).
- `evaluation_mode = "loso"` with a subject-level validation holdout.
- `val_split` is no longer a tiny within-subject fraction; we hold out whole subjects for validation.

In [4]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ---------------- Paths / run identity ----------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal-loso"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "loso_sjepa_prelocal",
    "config_note": "Cross-subject leave-one-subject-out S-JEPA PreLocal on Liu2024.",

    # ---------------- Dataset ----------------
    "subjects_to_use": None,          # None = all 50
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ---------------- Source-domain preprocessing ----------------
    "demean_mode": "baseline_window_mean",   # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],         # pre-MI instruction period as baseline

    # ---------------- MNE Raw-level preprocessing ----------------
    "reference_mode": "average",             # average, none
    "resample": True,
    "resample_sfreq": 128,
    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",
    "filter_phase": "zero",
    "filter_fir_design": "firwin",

    # ---------------- Windowing ----------------
    # MI cue starts at 2.0 s. 537 samples (~4.195 s) keeps S-JEPA tokenization compatibility.
    "mi_window_start_s": 2.0,
    "target_window_samples": 537,

    # ---------------- Normalization ----------------
    # Frozen encoder ("new") was trained on microvolt-scale data, so default to none to preserve scale.
    # For strategy="full", "train_channel_zscore" (fit on the train pool only) is worth a sweep
    # because cross-subject amplitude variance is large.
    "normalization_mode": "none",            # none, train_channel_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ---------------- Model / downstream strategy ----------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "random",    # from_pretrained, random
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                       # new, full
    "warmup_epochs": 10,                     # used only when strategy="full"

    # ---------------- Evaluation protocol (LOSO) ----------------
    "evaluation_mode": "loso",
    "n_val_subjects": 5,                     # subjects held out of the 49 for validation/early-stopping
    "loso_test_subjects": None,              # None = every subject is the test subject once; or a list

    # ---------------- Training hyperparameters ----------------
    "batch_size": 64,                        # large train pool -> larger batch is fine
    "n_epochs": 300,
    "early_stopping_patience": 30,
    "learning_rate": 5e-4,
    "optimizer_name": "adam",                # adam, adamw
    "weight_decay": 0.0,
    "checkpoint_metric": "valid_loss",       # valid_loss, valid_balanced_accuracy

    # ---------------- Diagnostics ----------------
    "collapse_threshold": 0.90,

    # ---------------- Reproducibility ----------------
    "seed": 2026,
    "set_seed": True,
}


In [5]:
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
TARGET_N_CLASSES = 2

EFFECTIVE_SFREQ = float(CONFIG["resample_sfreq"]) if CONFIG["resample"] else float(LIU_SOURCE_SFREQ)
WINDOW_SAMPLES = int(CONFIG["target_window_samples"])
TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / datetime.now().strftime("%Y%m%d_%H%M")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("Effective LOSO settings:")
print(f"  Channels:            {len(EEG_CHANNEL_NAMES)}")
print(f"  Effective sfreq:     {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start:     {CONFIG['mi_window_start_s']} s")
print(f"  Window samples:      {WINDOW_SAMPLES} ({TARGET_TRIAL_DURATION_S:.4f} s)")
print(f"  Evaluation:          {CONFIG['evaluation_mode']} | val subjects = {CONFIG['n_val_subjects']}")
print(f"  Strategy:            {CONFIG['strategy']} | pretrained = {CONFIG['pretrained_mode']}")
print(f"  Artifacts:           {ARTIFACT_DIR}")


Effective LOSO settings:
  Channels:            29
  Effective sfreq:     128.0 Hz
  MI window start:     2.0 s
  Window samples:      537 (4.1953 s)
  Evaluation:          loso | val subjects = 5
  Strategy:            new | pretrained = random
  Artifacts:           /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-source-mat-sjepa-prelocal-loso/20260609_1936


## 2.3 Reproducibility and device

In [6]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)

def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Device: {DEVICE} | base seed: {BASE_SEED}")


Device: mps | base seed: 2026


# 3. Load and prepare data

## 3.1 `.mat` loader

Same recursive-struct loader idea as the within-subject notebook: it finds the 3D data array and the
label vector regardless of whether the file exposes top-level `rawdata`/`labels` or a nested `eeg` struct.

In [7]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk(item, f"{prefix}{idx}")

def _normalize_rawdata_shape(arr, labels=None):
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got {arr.shape}")
    trial_axes = []
    if labels is not None:
        n = int(np.asarray(labels).size)
        trial_axes = [ax for ax, s in enumerate(arr.shape) if s == n]
    if not trial_axes:
        trial_axes = [ax for ax, s in enumerate(arr.shape) if s in (39, 40)]
    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)
    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw(name, arr):
    l = name.lower(); s = 0
    if "raw" in l or "data" in l: s += 10
    if arr.ndim == 3: s += 5
    if any(d in (39, 40) for d in arr.shape): s += 3
    if max(arr.shape) >= 3000: s += 2
    if "eeg" in l: s += 1
    return s

def _score_label(name, arr):
    l = name.lower(); flat = np.asarray(arr).ravel(); s = 0
    if "label" in l or "class" in l or l.split(".")[-1] in {"y", "labels"}: s += 10
    if flat.size in (39, 40): s += 3
    if flat.size <= 200 and set(np.unique(flat).astype(int)).issubset({0, 1, 2}): s += 2
    return s

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    arrays = [(n, np.asarray(v)) for n, v in _walk(mat)
              if isinstance(v, np.ndarray) and v.dtype != object]
    raws = [(_score_raw(n, a), n, a) for n, a in arrays if a.ndim == 3]
    labs = [(_score_label(n, a), n, a) for n, a in arrays if a.ndim in (1, 2)]
    if not raws or not labs:
        raise KeyError(f"Could not locate 3D data + labels in {path}")
    _, _, raw_arr = sorted(raws, key=lambda x: x[0], reverse=True)[0]
    _, _, lab_arr = sorted(labs, key=lambda x: x[0], reverse=True)[0]
    labels = np.asarray(lab_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels).astype(np.float64)
    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label/trial mismatch in {path}: {labels.shape} vs {rawdata.shape}")
    return rawdata, labels

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    u = set(np.unique(labels).tolist())
    if u.issubset({1, 2}):   # 1 = left, 2 = right  ->  0 = left, 1 = right
        return labels - 1
    if u.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(u)}")


## 3.2 Preprocessing pipeline (per subject)

Microvolts -> volts -> average reference -> resample -> FIR band-pass -> back to microvolts ->
crop the MI window. Same pipeline as the within-subject notebook; only the default window start
(2.0 s, MI-aligned) differs.

In [8]:
def make_liu_info(sfreq):
    info = mne.create_info(EEG_CHANNEL_NAMES, float(sfreq), ["eeg"] * len(EEG_CHANNEL_NAMES))
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def _to_volts(x):
    return np.asarray(x, np.float64) * (1e-6 if CONFIG["source_unit"].startswith("micro") else 1.0)

def _to_model_unit(x):
    return np.asarray(x, np.float64) * (1e6 if CONFIG["final_model_unit"].startswith("micro") else 1.0)

def preprocess_subject(rawdata, labels):
    X = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)   # trials x 29 x 4000
    n_trials = X.shape[0]

    # Source-domain baseline removal (pre-MI window) before going to MNE.
    if CONFIG["demean_mode"] == "baseline_window_mean":
        b0, b1 = CONFIG["baseline_window_s"]
        s0, s1 = int(round(b0 * LIU_SOURCE_SFREQ)), int(round(b1 * LIU_SOURCE_SFREQ))
        X = X - X[:, :, s0:s1].mean(axis=-1, keepdims=True)
    elif CONFIG["demean_mode"] == "trial_mean":
        X = X - X.mean(axis=-1, keepdims=True)

    # Continuous RawArray (trials concatenated along time), like the within-subject notebook.
    cont = _to_volts(X).transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    raw = mne.io.RawArray(cont, make_liu_info(LIU_SOURCE_SFREQ), verbose=False)

    if CONFIG["reference_mode"] == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
    if CONFIG["resample"]:
        raw.resample(float(CONFIG["resample_sfreq"]), verbose=False)
    if CONFIG["filter_enabled"]:
        raw.filter(l_freq=CONFIG["filter_low"], h_freq=CONFIG["filter_high"], method="fir",
                   phase=CONFIG["filter_phase"], fir_design=CONFIG["filter_fir_design"], verbose=False)

    eff = float(raw.info["sfreq"])
    data = _to_model_unit(raw.get_data())
    per_trial = data.shape[1] // n_trials
    data = data[:, :n_trials * per_trial]
    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, per_trial).transpose(1, 0, 2)

    start = int(round(CONFIG["mi_window_start_s"] * eff))
    stop = start + WINDOW_SAMPLES
    if stop > X_rs.shape[-1]:
        raise ValueError(f"Window [{start}:{stop}] exceeds trial length {X_rs.shape[-1]} at {eff} Hz")
    X_win = X_rs[:, :, start:stop]

    y = labels_to_zero_based(labels)
    return X_win.astype(np.float32), y.astype(np.int64)


## 3.3 Load and preprocess every subject once

In [9]:
MAT_FILES = find_source_mat_files(SOURCE_EXTRACT_DIR)
if not MAT_FILES:
    raise FileNotFoundError(f"No .mat files under {SOURCE_EXTRACT_DIR}")

use = None if CONFIG["subjects_to_use"] is None else {int(s) for s in CONFIG["subjects_to_use"]}
excl = {int(s) for s in CONFIG["exclude_subjects"]}

SUBJECT_DATA = {}   # sid -> (X, y)
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if (use is not None and sid not in use) or sid in excl:
        continue
    rawdata, labels = load_subject_mat(p)
    X, y = preprocess_subject(rawdata, labels)
    SUBJECT_DATA[sid] = (X, y)

ALL_SUBJECTS = sorted(SUBJECT_DATA.keys())
N_CH = len(EEG_CHANNEL_NAMES)
print(f"Loaded {len(ALL_SUBJECTS)} subjects.")
example_sid = ALL_SUBJECTS[0]
print(f"Example subject {example_sid}: X={SUBJECT_DATA[example_sid][0].shape}, "
      f"class counts={np.bincount(SUBJECT_DATA[example_sid][1]).tolist()}")
total_trials = sum(len(y) for _, y in SUBJECT_DATA.values())
print(f"Total trials across all subjects: {total_trials}  "
      f"(train pool per LOSO fold ~ {total_trials - len(SUBJECT_DATA[example_sid][1]):d})")


Loaded 50 subjects.
Example subject 1: X=(40, 29, 537), class counts=[20, 20]
Total trials across all subjects: 2000  (train pool per LOSO fold ~ 1960)


# 4. Model

## 4.1 Channel info, model build, trainable phases

`from_pretrained(..., strict=False)` loads the pretrained local encoder; `spatial_conv` and
`final_layer` are newly initialized. We verify the loaded parameter count is non-zero so a silent
load failure can't masquerade as a real result.

In [10]:
_INFO = make_liu_info(EFFECTIVE_SFREQ)
CH_NAMES = list(_INFO["ch_names"])
CHS_INFO = _INFO["chs"]

NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")

def build_model():
    kwargs = dict(n_chans=N_CH, chs_info=CHS_INFO, n_times=WINDOW_SAMPLES, n_outputs=TARGET_N_CLASSES)
    if CONFIG["pretrained_mode"] == "from_pretrained":
        model = SignalJEPA_PreLocal.from_pretrained(CONFIG["pretrained_repo_id"], **kwargs, strict=False)
    elif CONFIG["pretrained_mode"] == "random":
        model = SignalJEPA_PreLocal(**kwargs)
    else:
        raise ValueError("pretrained_mode must be 'from_pretrained' or 'random'.")
    return model

def set_trainable(model, phase):
    if phase == "full":
        for _, p in model.named_parameters():
            p.requires_grad = True
    else:  # "new" or warmup: only the new layers train
        for n, p in model.named_parameters():
            p.requires_grad = any(n.startswith(pre) for pre in NEW_LAYER_PREFIXES)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if n_train == 0:
        raise RuntimeError(f"No trainable parameters for phase={phase}")
    return n_train

# Sanity-check the pretrained load once.
_m = build_model()
_fe = sum(p.numel() for n, p in _m.named_parameters() if n.startswith("feature_encoder."))
print(f"Model built. feature_encoder params: {_fe:,}  (should be non-zero, ~1e4)")
del _m


Model built. feature_encoder params: 13,840  (should be non-zero, ~1e4)


# 5. LOSO datasets and splits

## 5.1 Dataset class, normalization, split builder

Validation is a **subject-level** holdout taken from the 49 training subjects (not a random slice of
pooled trials), so the early-stopping signal reflects cross-subject generalization. Normalization,
if enabled, is fit on the train pool only.

In [11]:
class ArrayDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = np.asarray(X, np.float32)
        self.y = np.asarray(y, np.int64)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], int(self.y[i])

def fit_normalizer(X_train):
    mode = CONFIG["normalization_mode"]
    eps = CONFIG["normalization_eps"]
    if mode == "train_channel_zscore":
        mean = X_train.mean(axis=(0, 2), keepdims=True)
        std = X_train.std(axis=(0, 2), keepdims=True)
        std = np.where(std < eps, 1.0, std)
        return {"mode": mode, "mean": mean.astype(np.float32), "std": std.astype(np.float32)}
    return {"mode": mode}

def apply_normalizer(X, state):
    mode = state["mode"]
    if mode == "none":
        return X
    if mode == "train_channel_zscore":
        return ((X - state["mean"]) / state["std"]).astype(np.float32)
    if mode == "trial_channel_zscore":
        mean = X.mean(axis=2, keepdims=True)
        std = X.std(axis=2, keepdims=True)
        std = np.where(std < CONFIG["normalization_eps"], 1.0, std)
        return ((X - mean) / std).astype(np.float32)
    raise ValueError(f"Unknown normalization_mode={mode}")

def stack_subjects(sids):
    Xs = [SUBJECT_DATA[s][0] for s in sids]
    ys = [SUBJECT_DATA[s][1] for s in sids]
    return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0)

def build_loso_split(test_sid):
    others = [s for s in ALL_SUBJECTS if s != test_sid]
    rng = np.random.default_rng(BASE_SEED + int(test_sid))
    rng.shuffle(others)
    n_val = int(CONFIG["n_val_subjects"])
    val_sids = sorted(others[:n_val])
    train_sids = sorted(others[n_val:])

    X_train, y_train = stack_subjects(train_sids)
    X_val, y_val = stack_subjects(val_sids)
    X_test, y_test = SUBJECT_DATA[test_sid]

    norm = fit_normalizer(X_train)
    X_train = apply_normalizer(X_train, norm)
    X_val = apply_normalizer(X_val, norm)
    X_test = apply_normalizer(np.asarray(X_test, np.float32), norm)

    return (ArrayDataset(X_train, y_train), ArrayDataset(X_val, y_val),
            ArrayDataset(X_test, y_test), y_test, train_sids, val_sids)


# 6. Training and evaluation runner

## 6.1 Classifier builder + single-fold runner

Uses `predefined_split` so skorch validates on the held-out subjects each epoch and `EarlyStopping`
restores the best checkpoint. For `strategy="full"` we warm up the new layers first, then unfreeze.

In [12]:
def make_callbacks():
    monitor = "valid_loss" if CONFIG["checkpoint_metric"] == "valid_loss" else "valid_acc"
    lower = CONFIG["checkpoint_metric"] == "valid_loss"
    return [("early_stopping", EarlyStopping(monitor=monitor, lower_is_better=lower,
                                             patience=int(CONFIG["early_stopping_patience"]),
                                             load_best=True))]

def build_classifier(model, valid_ds, max_epochs):
    opt = torch.optim.Adam if CONFIG["optimizer_name"] == "adam" else torch.optim.AdamW
    return EEGClassifier(
        model,
        criterion=torch.nn.CrossEntropyLoss,
        optimizer=opt,
        optimizer__weight_decay=float(CONFIG["weight_decay"]),
        lr=float(CONFIG["learning_rate"]),
        batch_size=int(CONFIG["batch_size"]),
        max_epochs=int(max_epochs),
        train_split=predefined_split(valid_ds),
        callbacks=make_callbacks(),
        device=DEVICE,
        classes=list(range(TARGET_N_CLASSES)),
        iterator_train__shuffle=True,
        iterator_train__num_workers=0,
        iterator_valid__num_workers=0,
    )

def collapse_ratio(y_pred):
    h = np.bincount(np.asarray(y_pred, int), minlength=TARGET_N_CLASSES)
    return float(h.max() / h.sum()) if h.sum() else 0.0

def run_one_loso(test_sid):
    if CONFIG["set_seed"]:
        seed_everything(BASE_SEED)
    train_ds, val_ds, test_ds, y_test, train_sids, val_sids = build_loso_split(test_sid)

    model = build_model()
    if CONFIG["strategy"] == "new":
        set_trainable(model, "new")
        clf = build_classifier(model, val_ds, CONFIG["n_epochs"])
        clf.fit(train_ds, y=None)
    elif CONFIG["strategy"] == "full":
        set_trainable(model, "new")
        clf = build_classifier(model, val_ds, int(CONFIG["warmup_epochs"]))
        clf.fit(train_ds, y=None)
        set_trainable(clf.module_, "full")
        clf.set_params(callbacks=make_callbacks(), max_epochs=int(CONFIG["n_epochs"]))
        clf.initialize_optimizer()
        clf.initialize_callbacks()
        clf.partial_fit(train_ds, y=None)
    else:
        raise ValueError("strategy must be 'new' or 'full'.")

    y_pred = clf.predict(test_ds)
    return {
        "test_subject": int(test_sid),
        "n_train": len(train_ds),
        "n_val": len(val_ds),
        "n_test": len(test_ds),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "confusion": confusion_matrix(y_test, y_pred, labels=range(TARGET_N_CLASSES)).tolist(),
        "collapse_ratio": collapse_ratio(y_pred),
        "y_true": np.asarray(y_test, int).tolist(),
        "y_pred": np.asarray(y_pred, int).tolist(),
        "val_subjects": val_sids,
    }


# 7. Run the LOSO loop

In [13]:
test_subjects = CONFIG["loso_test_subjects"] or ALL_SUBJECTS
RESULTS = []
for i, sid in enumerate(test_subjects, 1):
    res = run_one_loso(sid)
    RESULTS.append(res)
    print(f"[{i:2d}/{len(test_subjects)}] subject {sid:2d}  "
          f"BA={res['balanced_accuracy']*100:5.1f}%  "
          f"acc={res['accuracy']*100:5.1f}%  "
          f"collapse={res['collapse_ratio']:.2f}  "
          f"(train={res['n_train']}, val={res['n_val']}, test={res['n_test']})")

with open(ARTIFACT_DIR / "loso_results.json", "w") as f:
    json.dump(RESULTS, f, indent=2)


  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.7066       0.5200        0.6919  1.2051
      2        0.6861       0.5450        0.6871  0.4420
      3        0.6723       0.5650        0.6858  0.4644
      4        0.6596       0.5600        0.6876  0.4468
      5        0.6502       0.5250        0.6972  0.4376
      6        0.6430       0.5500        0.6888  0.4862
      7        0.6332       0.5450        0.6909  0.4815
      8        0.6240       0.4900        0.6924  0.4619
      9        0.6187       0.5200        0.7031  0.4675
     10        0.6115       0.5200        0.6948  0.4668
     11        0.6046       0.5150        0.6970  0.4579
     12        0.5973       0.5300        0.6996  0.3764
     13        0.5941       0.5100        0.7056  0.4071
     14        0.5893       0.5250        0.7094  0.4552
     15        0.5836       0.5350        0.7088  0.4688
     16        0.5776       0.5

# 8. Aggregate results

In [14]:
ba = np.array([r["balanced_accuracy"] for r in RESULTS])
collapse_flag = np.array([r["collapse_ratio"] >= CONFIG["collapse_threshold"] for r in RESULTS])

y_true_all = np.concatenate([r["y_true"] for r in RESULTS])
y_pred_all = np.concatenate([r["y_pred"] for r in RESULTS])
global_ba = balanced_accuracy_score(y_true_all, y_pred_all)
cm = confusion_matrix(y_true_all, y_pred_all, labels=range(TARGET_N_CLASSES))

print("=" * 60)
print(f"LOSO over {len(RESULTS)} subjects | strategy={CONFIG['strategy']} | "
      f"pretrained={CONFIG['pretrained_mode']}")
print(f"  Mean per-subject balanced accuracy: {ba.mean()*100:.2f}%  (SD {ba.std()*100:.2f})")
print(f"  Global pooled balanced accuracy:    {global_ba*100:.2f}%")
print(f"  Collapsed subjects (>= {CONFIG['collapse_threshold']:.0%} one class): "
      f"{int(collapse_flag.sum())}/{len(RESULTS)}")
print(f"  Pooled confusion [rows=true 0/1, cols=pred 0/1]:\n{cm}")
print("=" * 60)

summary_df = pd.DataFrame([
    {"subject": r["test_subject"],
     "balanced_acc_%": round(r["balanced_accuracy"] * 100, 1),
     "acc_%": round(r["accuracy"] * 100, 1),
     "collapse_ratio": round(r["collapse_ratio"], 2)}
    for r in RESULTS
]).sort_values("subject").reset_index(drop=True)
summary_df.to_csv(ARTIFACT_DIR / "loso_per_subject.csv", index=False)
summary_df


LOSO over 50 subjects | strategy=new | pretrained=random
  Mean per-subject balanced accuracy: 49.75%  (SD 8.96)
  Global pooled balanced accuracy:    49.75%
  Collapsed subjects (>= 90% one class): 0/50
  Pooled confusion [rows=true 0/1, cols=pred 0/1]:
[[568 432]
 [573 427]]


,subject,balanced_acc_%,acc_%,collapse_ratio
0,1,50.0,50.0,0.65
1,2,42.5,42.5,0.53
2,3,52.5,52.5,0.53
3,4,42.5,42.5,0.68
4,5,47.5,47.5,0.57
5,6,42.5,42.5,0.53
6,7,55.0,55.0,0.70
7,8,67.5,67.5,0.53
8,9,22.5,22.5,0.53
9,10,45.0,45.0,0.55


## 8.1 Notes / next knobs to sweep

- **Compare against the random control:** rerun with `pretrained_mode="random"`. The interesting
  number is the pretrained-minus-random gap under this cross-subject regime.
- **`strategy="full"`** usually helps more once there's enough data; try it after `new`.
- **Normalization:** with cross-subject data, sweep `normalization_mode="train_channel_zscore"`
  (and consider `strategy="full"` so the frozen encoder isn't fed an off-scale input).
- **Window:** `mi_window_start_s` is MI-aligned at 2.0 s; sweep 1.5-2.5 s if needed.
- If collapse is still present here (it shouldn't be, with ~1,960 train trials + a real val set),
  that points to an optimization issue rather than data scarcity.